In [1]:
!pip install PyPortfolioOpt
!pip install scikit-learn
!pip install stable-baselines3
!pip install gymnasium
!pip install pypfopt
!pip install tqdm
!pip install torch
!pip install torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.1/220.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

Reference:

First Prompt(Jobina): I am working on a project. It takes data from YahooFinance and train the DDPG model for portfolio allocation and should accept investment amount as input and should provide the list of tickers that one can buy, hold or sell. Including when to buy hold and sell, how much shares. Help me with the code.

Last Prompt(Jobina) : "Generate a Python script that implements a portfolio optimization system using Deep Reinforcement Learning (DRL).
The script should be structured as follows:

A custom Gymnasium environment (PortfolioTradingEnv) that simulates stock portfolio management.
A data iterator (StockPortfolioIterator) to process historical stock data.
A DDPG model from stable-baselines3 for portfolio allocation.
A training function that optimizes the DRL agent.
An evaluation function to assess model performance using metrics like Sharpe Ratio, Total Return, and Max Drawdown.
A backtesting function that simulates the trained model’s trading strategy.
A portfolio recommender system that provides AI-driven stock allocation recommendations.
A main script (if __name__ == "__main__") that supports CLI execution modes:
"train" → Train a new model.
"evaluate" → Evaluate a trained model.
"backtest" → Run historical strategy simulation.
"recommend" → Generate AI-driven investment recommendations.
The script should support logging results to JSON and storing models as checkpoint files." I will share you the code which I have developed till now so you can pass paramter accordinlgy.

MemoryError: Unable to allocate 448. MiB for an array with shape (1000, 1, 117501) and data type float32 Help me resolve the error.






First Prompt(Anand): Help me configure gynasium environment - training AI-based portfolio managers, enabling them to learn optimal portfolio rebalancing strategies over time.

Last Prompt(Anand): Evaluate a trained Deep Reinforcement Learning (DRL) model on unseen stock market data to assess its performance. The evaluation should include:

Performance Metrics Calculation:
Sharpe Ratio (risk-adjusted return).
Max Drawdown (largest peak-to-trough decline).
Total Return (overall portfolio growth).
Comparison Against Benchmark:
Compare portfolio performance with a market index (e.g., S&P 500).
Analyze risk-adjusted returns.
Evaluation Process:
Load the model and test it on new stock data not seen during training.
Track portfolio value over time and compute daily returns.
Analyze win rate and volatility.
Output:
Print a summary of performance metrics.
Visualize results using charts (e.g., equity curve, drawdown plot, returns histogram).
Ensure robustness by handling potential errors (e.g., missing data, extreme price movements).





First Prompt(Jeffin): Help me with the code for back testing.

Last Prompt(Jeffin): "I am encountering a MemoryError while trying to allocate a large NumPy array with shape (1000, 1, 117501) and data type float32, which requires 448 MiB of memory.
Please provide an optimized solution to reduce memory usage while ensuring efficient data processing.
The solution should include:

Reducing dataset size (e.g., limiting stocks, lookback period, or years of data).
Optimizing data types (e.g., converting float32 to float16).
Batch processing instead of loading everything into memory at once.
Using NumPy memory-mapped files (mmap) for large datasets.
Increasing swap memory as a last resort.
The goal is to fix the MemoryError while maintaining performance in a stock portfolio optimization system.





In [2]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DDPG
from stable_baselines3.common.vec_env import DummyVecEnv
from sklearn.preprocessing import StandardScaler
import torch
from datetime import datetime
from stable_baselines3.common.noise import NormalActionNoise

# Stock Portfolio Data Iterator for Deep Reinforcement Learning

The StockPortfolioIterator class is a custom iterator designed for loading, processing, and iterating over historical stock market data for portfolio management using Deep Reinforcement Learning (DRL). It prepares stock price data by applying preprocessing techniques such as filtering based on trading history, computing technical indicators, and normalizing features.

This iterator enables efficient batch retrieval of stock price data over a lookback window, making it suitable for training reinforcement learning models that require sequential financial data. Key functionalities include:

Data Loading & Preprocessing: Reads stock market data from a CSV file, filters stocks based on minimum trading history and trading volume, and prepares a structured dataset.

Feature Engineering: Computes essential technical indicators like Moving Averages, RSI, MACD, Momentum, and Volatility.

Data Normalization: Scales numerical features for consistent input representation.

Efficient Data Storage & Access: Stores preprocessed data in a dictionary format for quick retrieval during training.

Iterative Batch Processing: Provides a sequential data stream with a defined lookback window for reinforcement learning agents.

This iterator simplifies data handling for stock portfolio optimization models and ensures that financial data is structured effectively for deep learning-based portfolio allocation strategies.

In [3]:
class StockPortfolioIterator:
    def __init__(self, file_path: str, lookback_window: int = 30, min_history: int = 100, max_stocks: int = 100, train_years: int = 20):

        print(f"Loading data from {file_path}...")
        self.data = pd.read_csv(file_path)
        self.data['date'] = pd.to_datetime(self.data['date'])

        # Only use recent data (last few years)
        cutoff_date = self.data['date'].max() - pd.DateOffset(years=train_years)
        self.data = self.data[self.data['date'] >= cutoff_date]

        self.data = self.data.sort_values(['date', 'tic'])

        # Filter stocks with enough history
        stock_counts = self.data.groupby('tic').size()
        valid_stocks = stock_counts[stock_counts >= min_history].index

        # Select stocks with highest trading volume (more likely to be liquid)
        if len(valid_stocks) > max_stocks:
            avg_volumes = self.data[self.data['tic'].isin(valid_stocks)].groupby('tic')['volume'].mean()
            valid_stocks = avg_volumes.nlargest(max_stocks).index.tolist()

        self.data = self.data[self.data['tic'].isin(valid_stocks)]
        self.unique_dates = sorted(self.data['date'].unique())
        self.tickers = sorted(self.data['tic'].unique())
        self.num_stocks = len(self.tickers)
        self.lookback = lookback_window
        self.num_features = 7

        # Create ticker lookup for faster access
        self.ticker_indices = {ticker: i for i, ticker in enumerate(self.tickers)}

        self._add_technical_indicators()
        self._scale_features()

        # Store data in a more efficient format
        self.prepare_training_data()

        print(f"Loaded {self.num_stocks} stocks with {len(self.unique_dates)} trading days")
        print(f"Date range: {self.unique_dates[0]} to {self.unique_dates[-1]}")

    def _add_technical_indicators(self):

        print("Adding technical indicators...")
        indicators = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        # Process one ticker at a time to avoid memory issues
        for tic in self.tickers:
            mask = self.data['tic'] == tic
            stock_data = self.data.loc[mask].copy()

            # Calculate technical indicators
            stock_data['Returns'] = stock_data['close'].pct_change()
            stock_data['Price_MA'] = stock_data['close'].rolling(window=20, min_periods=1).mean()
            stock_data['Momentum'] = stock_data['close'].pct_change(periods=10)
            stock_data['Volume_MA'] = stock_data['volume'].rolling(window=10, min_periods=1).mean()

            # RSI Calculation
            delta = stock_data['close'].diff()
            gain = (delta.clip(lower=0)).rolling(window=14, min_periods=1).mean()
            loss = (-delta.clip(upper=0)).rolling(window=14, min_periods=1).mean()
            rs = gain / (loss + 1e-8)
            stock_data['RSI'] = 100 - (100 / (1 + rs))

            # MACD Calculation
            exp1 = stock_data['close'].ewm(span=12, adjust=False).mean()
            exp2 = stock_data['close'].ewm(span=26, adjust=False).mean()
            stock_data['MACD'] = exp1 - exp2

            # Volatility
            stock_data['Volatility'] = stock_data['Returns'].rolling(window=30, min_periods=1).std()

            # Update the main dataframe
            self.data.loc[mask, indicators] = stock_data[indicators]

        # Fill missing values
        self.data = self.data.fillna(0)
        print("Technical indicators added successfully")

    def _scale_features(self):
        print("Scaling features...")
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        for tic in self.tickers:
            mask = self.data['tic'] == tic
            scaler = StandardScaler()
            self.data.loc[mask, feature_columns] = scaler.fit_transform(self.data.loc[mask, feature_columns])

    def prepare_training_data(self):
        print("Preparing training data...")
        # Pre-allocate arrays for features and prices
        self.all_features = {}
        self.all_prices = {}
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        # Process data by date for efficient access during training
        for date_idx, date in enumerate(self.unique_dates):
            date_mask = self.data['date'] == date
            date_data = self.data[date_mask]

            features = np.zeros((self.num_stocks, self.num_features))
            prices = np.zeros(self.num_stocks)

            for _, row in date_data.iterrows():
                ticker_idx = self.ticker_indices.get(row['tic'])
                if ticker_idx is not None:
                    features[ticker_idx] = row[feature_columns].values
                    prices[ticker_idx] = row['close']

            self.all_features[date] = features
            self.all_prices[date] = prices

    def __iter__(self):
        """Initialize iterator"""
        self.current_idx = self.lookback
        return self

    def __next__(self):
        """Get next batch of data"""
        if self.current_idx >= len(self.unique_dates):
            raise StopIteration

        current_date = self.unique_dates[self.current_idx]
        window_dates = self.unique_dates[self.current_idx - self.lookback:self.current_idx]

        # Get lookback window features
        features = np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32)

        for i, date in enumerate(window_dates):
            if date in self.all_features:
                features[:, i, :] = self.all_features[date]

        prices = self.all_prices[current_date]

        self.current_idx += 1
        return {
            'features': features,
            'prices': prices,
            'date': current_date,
            'tickers': self.tickers
        }

    def reset(self):
        """Reset the iterator"""
        self.current_idx = self.lookback
        return self

# Portfolio Recommender Using Deep Reinforcement Learning

Description:
The PortfolioRecommender class is an AI-powered financial tool that suggests optimal stock allocations using a trained Deep Deterministic Policy Gradient (DDPG) model. It processes the latest stock market data, computes relevant features, and generates investment recommendations based on reinforcement learning strategies.

This class is particularly useful for beginner investors looking to allocate a predefined investment amount into a diversified stock portfolio while keeping a portion in cash. The recommendations are generated by analyzing recent stock performance and computing optimal portfolio weights.

Key Features:
DDPG Model Integration: Loads a pre-trained Deep Reinforcement Learning (DRL) model for portfolio optimization.
Stock Selection: Filters the top liquid stocks based on trading volume to ensure reliable investments.
Feature Engineering: Computes key technical indicators, including returns, volatility, and momentum to ensure model consistency.
Smart Allocation Strategy:
Predicts portfolio weights for stocks and cash allocation.
Normalizes investment allocations to sum up to 100% of the input capital.
Ensures risk-adjusted diversification by using historical trends.
How It Works:
Loads Market Data → Reads the latest stock data and selects the top stocks based on liquidity.
Prepares Features → Extracts the most relevant technical indicators to match training data.
Predicts Portfolio Weights → Uses the DDPG model to determine the optimal stock allocation.
Generates Investment Plan → Converts model output into real-world investment recommendations:
Percentage allocation per stock
Amount invested per stock (in CAD)
Number of shares to buy
Remaining cash reserve
Ideal Use Case:
This tool is designed for investors with no prior trading experience who want AI-driven insights into portfolio allocation. It ensures an intelligent and data-driven approach to investing in stocks while minimizing risk and maximizing returns over time.

In [4]:
class PortfolioRecommender:
    def __init__(self, model_path, data_path, max_stocks=100, lookback=30):
        """Initialize the portfolio recommender with the trained DDPG model and latest stock data."""
        self.model = DDPG.load(model_path)  # Load DDPG model
        self.max_stocks = max_stocks
        self.lookback = lookback

        # Load and preprocess stock data
        data = pd.read_csv(data_path)
        data['date'] = pd.to_datetime(data['date'])

        # Get the most recent trading date
        self.latest_date = data['date'].max()

        # Select top stocks by trading volume (same filtering as in training)
        recent_data = data[data['date'] >= (self.latest_date - pd.DateOffset(days=30))]
        avg_volumes = recent_data.groupby('tic')['volume'].mean()
        top_tickers = avg_volumes.nlargest(max_stocks).index.tolist()

        # Get the ordered list of tickers (must match training data order)
        self.tickers = sorted(top_tickers)

        # Store latest closing prices for selected stocks
        latest_data = data[data['date'] == self.latest_date]
        self.latest_prices = {row['tic']: row['close'] for _, row in latest_data.iterrows()
                              if row['tic'] in self.tickers}

        # Compute technical indicators (ensures feature consistency)
        self._prepare_features(data)

    def _prepare_features(self, data):
        """Prepares feature matrix for the latest available stock data."""
        filtered_data = data[data['tic'].isin(self.tickers)]

        # Get the most recent dates for feature calculation
        recent_dates = sorted(filtered_data['date'].unique())[-self.lookback:]
        recent_data = filtered_data[filtered_data['date'].isin(recent_dates)]

        # Initialize the feature array
        features = np.zeros((self.max_stocks, self.lookback, 7), dtype=np.float32)

        for i, ticker in enumerate(self.tickers):
            if i >= self.max_stocks:
                break

            ticker_data = recent_data[recent_data['tic'] == ticker].sort_values('date')
            if len(ticker_data) == self.lookback:
                # Create feature array (must match training data features)
                ticker_features = np.zeros((self.lookback, 7))
                ticker_features[:, 0] = ticker_data['close'].pct_change().fillna(0).values  # Returns
                ticker_features[:, 1] = ticker_data['volume'].values / ticker_data['volume'].mean()  # Normalized Volume
                ticker_features[:, 2] = ticker_data['close'].values / ticker_data['close'].mean()  # Normalized Price
                ticker_features[:, 3] = 50  # Placeholder for RSI
                ticker_features[:, 4] = 0   # Placeholder for MACD
                ticker_features[:, 5] = ticker_data['close'].pct_change().rolling(5).std().fillna(0).values  # Volatility
                ticker_features[:, 6] = ticker_data['close'].pct_change(5).fillna(0).values  # Momentum

                features[i] = ticker_features

        self.recent_features = features

    def recommend_portfolio(self, amount_cad):
        """Generates stock allocation recommendations based on the trained DDPG model."""

        # Use the latest computed features for prediction
        action, _ = self.model.predict(self.recent_features, deterministic=True)

        # Normalize action values to sum to 1
        action = np.clip(action, 0, 1)
        if action.sum() > 0:
            action /= action.sum()

        # Allocate cash and stocks
        allocations = {}
        cash_allocation = action[0] * amount_cad  # First action is cash allocation

        for i, ticker in enumerate(self.tickers):
            if i >= len(self.tickers) or i+1 >= len(action):
                continue

            allocation = action[i+1] * amount_cad  # Skip first index (cash)
            if allocation > 0 and ticker in self.latest_prices:
                price = self.latest_prices[ticker]
                shares = allocation / price if price > 0 else 0

                allocations[ticker] = {
                    "allocation_percent": float(action[i+1] * 100),
                    "allocation_cad": float(allocation),
                    "price_per_share": float(price),
                    "shares": float(shares)
                }

        return {
            "cash_percent": float(action[0] * 100),
            "cash_amount": float(cash_allocation),
            "stock_allocations": allocations,
            "total_amount": float(amount_cad),
            "recommendation_date": str(self.latest_date)
        }



# Portfolio Trading Environment for Deep Reinforcement Learning

The PortfolioTradingEnv class defines a custom OpenAI Gym environment designed for training Deep Reinforcement Learning (DRL) agents (such as DDPG) to manage a stock portfolio. It simulates real-world stock trading conditions by allowing an agent to allocate capital across multiple stocks, adjusting its holdings dynamically while accounting for transaction costs.

This environment is crucial for training DRL-based portfolio optimization models, enabling them to learn investment strategies that maximize returns while minimizing risks.

Key Features:
State Representation:
Observations consist of technical indicators over a lookback window for multiple stocks.
Action Space:
A continuous action space where the model assigns portfolio weights across stocks and cash.
Portfolio Management Logic:
The agent decides how to allocate capital among stocks and cash.
The system executes trades based on the portfolio allocation.
Transaction costs are deducted based on trade volume.
Reward Mechanism:
The reward function is based on portfolio value appreciation after executing trades, penalizing transaction costs.
Episode Termination:
The episode ends when the dataset is exhausted, ensuring the model learns from historical market patterns.
How It Works:
Initialization (__init__):

Loads stock data using a data iterator.
Defines the action space (portfolio allocation) and observation space (historical stock features).
Sets up initial cash balance and stock holdings.
Reset (reset):

Resets cash and holdings to starting conditions.
Fetches the first set of stock data from the iterator.
Step (step):

Receives a portfolio allocation decision from the DRL agent.
Normalizes actions to ensure total allocation sums to 100%.
Computes new stock holdings based on allocation.
Deducts transaction costs for portfolio adjustments.
Computes the new portfolio value and reward.
Fetches the next trading day’s data.
Determines if the episode has ended.

Ideal Use Case:
This environment is designed for training AI-based portfolio managers, enabling them to learn optimal portfolio rebalancing strategies over time. It is particularly useful for automated trading systems and hedge funds aiming to improve portfolio performance using AI-driven decision-making.

In [5]:
class PortfolioTradingEnv(gym.Env):
    def __init__(self, data_iterator, initial_cash=10000, transaction_cost=0.001):
        super().__init__()
        self.data_iterator = data_iterator
        self.initial_cash = initial_cash
        self.transaction_cost = transaction_cost
        self.num_stocks = data_iterator.num_stocks
        self.lookback = data_iterator.lookback
        self.num_features = data_iterator.num_features
        self.episode_reward = 0  # Track total reward
        self.episode_steps = 0  # Track number of steps
        self.debug_info = {}  # Store debug information

        # Action space: [cash allocation, stock1 allocation, ..., stockN allocation]
        self.action_space = spaces.Box(
            low=0,
            high=1,
            shape=(self.num_stocks + 1,),
            dtype=np.float32
        )

        # Observation space: (num_stocks, lookback, num_features)
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.num_stocks, self.lookback, self.num_features),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        """Reset the environment"""
        # Reset the data iterator
        self.data_iterator.reset()
        self.cash = self.initial_cash
        self.holdings = np.zeros(self.num_stocks)
        self.episode_reward = 0  # Reset total reward
        self.episode_steps = 0  # Reset step counter
        self.debug_info = {}  # Reset debug info

        try:
            self.current_step = next(self.data_iterator)
            return self._get_observation(), {}
        except StopIteration:
            # Handle the case where there's no more data
            return np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32), {}

    def _get_observation(self):
        """Get the current observation"""
        return self.current_step['features'].astype(np.float32)

    def step(self, action):
      """Take a step in the environment"""
      self.episode_steps += 1

      # Handle case where action is a scalar (possible in vector environments)
      if np.isscalar(action):
        action = np.array([action])  # Convert to array with single element

      # Ensure action has correct shape
      if action.shape != (self.num_stocks + 1,):
        if len(action.shape) > 1:
            # If action is a batch (from vector env), take the first one
            action = action[0]
        else:
            # If action is still wrong shape, create a default action (all cash)
            print(f"Warning: Invalid action shape {action.shape}, expected {(self.num_stocks + 1,)}")
            action = np.zeros(self.num_stocks + 1)
            action[0] = 1.0  # All cash

      # Normalize action
      action = np.clip(action, 0, 1)
      action_sum = action.sum()
      if action_sum > 0:
        action /= action_sum
      else:
        # Handle the case where all actions are zero
        action[0] = 1.0  # Set cash allocation to 100%

      # Store action for debugging
      if self.episode_steps % 100 == 0:
        self.debug_info['action'] = action.copy()

      current_prices = self.current_step['prices']
      portfolio_value = self.cash + np.sum(self.holdings * current_prices)

      # Calculate target allocations
      target_values = action[1:] * portfolio_value
      current_values = self.holdings * current_prices
      delta_values = target_values - current_values

      # Apply transaction costs
      transaction_costs = np.abs(delta_values).sum() * self.transaction_cost

      # Update holdings and cash
      for i in range(self.num_stocks):
        if current_prices[i] > 0:  # Avoid division by zero
            self.holdings[i] += delta_values[i] / current_prices[i]

      self.cash = portfolio_value - np.sum(self.holdings * current_prices) - transaction_costs

      # Store current portfolio value for reward calculation
      old_portfolio_value = portfolio_value

      try:
        # Move to next time step
        self.current_step = next(self.data_iterator)
        new_prices = self.current_step['prices']
        done = False
      except StopIteration:
        new_prices = current_prices  # Use current prices if no more data
        done = True

      # Calculate reward (daily return)
      new_portfolio_value = self.cash + np.sum(self.holdings * new_prices)
      reward = (new_portfolio_value / old_portfolio_value) - 1

      # Apply penalty for excessive trading
      reward -= transaction_costs / old_portfolio_value

      # Store debugging information
      if self.episode_steps % 100 == 0 or done:
        self.debug_info.update({
            'portfolio_value_before': old_portfolio_value,
            'portfolio_value_after': new_portfolio_value,
            'transaction_costs': transaction_costs,
            'reward': reward,
            'cash': self.cash,
            'holdings_sum': np.sum(self.holdings * new_prices)
        })
        print(f"Step {self.episode_steps}: Reward = {reward:.6f}, "
              f"Portfolio Value: {new_portfolio_value:.2f}, "
              f"Transaction Costs: {transaction_costs:.2f}")

      # Update total reward
      self.episode_reward += float(reward)

      return (
        self._get_observation() if not done else np.zeros_like(self.observation_space.sample()),
        float(reward),
        done,
        False,
        {
            "portfolio_value": new_portfolio_value,
            "total_reward": self.episode_reward,
            "transaction_costs": transaction_costs,
            "cash": self.cash,
            "holdings_value": np.sum(self.holdings * new_prices)
        }
    )

# model evaluation

Model Evaluation Function for Portfolio Trading
Description:
The evaluate_model function is designed to assess the performance of a trained Deep Reinforcement Learning (DRL) model (such as DDPG) in a simulated stock trading environment. It evaluates how well the model manages a stock portfolio over a validation dataset, calculating key performance metrics like total rewards, portfolio growth, and annualized returns.

This function is essential for validating the profitability and robustness of the trained AI trading model before deploying it in live market conditions.

Key Features:
Validation Dataset:
Uses recent stock data (past 5 years) for evaluation.
Selects top liquid stocks based on trading volume.
Performance Tracking:
Tracks total rewards (cumulative profit/loss).
Logs final portfolio value at the end of each episode.
Computes annualized returns based on trading performance.
Multiple Episodes for Stability:
Evaluates model performance over multiple episodes (default: 10).
Helps measure consistency and reliability of the strategy.
Performance Metrics:
Average total reward across episodes.
Average final portfolio value (growth from initial cash).
Success rate (percentage of episodes with positive returns).
How It Works:
Initialize Evaluation Environment:

Loads validation stock data.
Creates a StockPortfolioIterator for dataset processing.
Sets up the PortfolioTradingEnv to simulate trading.
Run Model Across Multiple Episodes:

Resets the environment at the start of each episode.
Executes trades using the trained model's predicted actions.
Tracks portfolio growth and rewards.
Calculate Performance Metrics:

Computes final portfolio value for each episode.
Derives annualized return using trading duration.
Computes average performance statistics over multiple episodes.
Print Evaluation Summary:

Displays per-episode performance.
Summarizes overall average reward, portfolio value, and success rate.
Ideal Use Case:
This function is ideal for quantitative finance researchers, algorithmic traders, and hedge funds who want to evaluate an AI-powered portfolio optimization model before deploying it in real-world trading.

In [6]:
def evaluate_model(model, data_path, max_stocks=100, lookback=30, episodes=10):
    """Evaluate the model on a validation dataset"""
    print(f"Evaluating model performance...")

    eval_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=5  # Use more recent data for evaluation
    )

    eval_env = PortfolioTradingEnv(eval_iter)

    total_rewards = []
    portfolio_values = []

    for ep in range(episodes):
        obs, _ = eval_env.reset()
        done = False
        episode_reward = 0
        initial_value = eval_env.initial_cash
        step_count = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info = eval_env.step(action)
            episode_reward += reward
            step_count += 1

            if done:
                final_value = info["portfolio_value"]
                portfolio_values.append(final_value)

                # Calculate annualized return
                if step_count > 0:
                    days = step_count
                    annualized_return = ((final_value / initial_value) ** (252 / days) - 1) * 100
                else:
                    annualized_return = 0

                print(f"Episode {ep+1}: Return = {episode_reward:.4f}, "
                      f"Steps = {step_count}, "
                      f"Final Value = ${final_value:.2f}, "
                      f"Annualized Return = {annualized_return:.2f}%")

        total_rewards.append(episode_reward)

    # Calculate summary statistics
    avg_reward = np.mean(total_rewards)
    avg_final_value = np.mean(portfolio_values)
    success_rate = np.sum(np.array(total_rewards) > 0) / len(total_rewards) * 100

    print(f"\nEvaluation Results (over {episodes} episodes):")
    print(f"Average Total Reward: {avg_reward:.4f}")
    print(f"Average Final Portfolio Value: ${avg_final_value:.2f}")
    print(f"Success Rate (positive return): {success_rate:.2f}%")

    return total_rewards, portfolio_values

# Deep Reinforcement Learning Model Training for Portfolio Optimization

The train_model function is a structured pipeline for training a Deep Deterministic Policy Gradient (DDPG) model to optimize stock portfolio allocations. It uses historical stock data to train the AI model to make intelligent investment decisions by dynamically rebalancing assets while considering market trends and transaction costs.

This function ensures a robust training process by incorporating staged training with checkpoints, random policy evaluation, and intermediate model evaluations to track performance improvements.

Key Features:
Stock Data Processing:
Uses StockPortfolioIterator to preprocess stock market data.
Filters high-volume stocks to ensure liquid investments.
Reinforcement Learning Environment:
Utilizes PortfolioTradingEnv to simulate trading.
Defines an action space for portfolio allocation across stocks and cash.
DDPG Model Configuration:
Implements a policy-based reinforcement learning model using an actor-critic architecture.
Applies normal action noise for exploration.
Trains on GPU if available for faster processing.
Logs training progress using TensorBoard.
Training Pipeline:
Initial random policy evaluation to establish a performance baseline.
Incremental training over five stages, saving checkpoints at each stage.
Quick policy evaluation after each training phase.
Final model saving after training completion.
Error Handling & Robustness:
Captures potential training errors and ensures continuation.
Implements failsafe mechanisms for saving checkpoints.
How It Works:
Prepares Training Data

Loads stock price history and technical indicators.
Filters stocks based on trading volume and history.
Creates Reinforcement Learning Environment

Initializes PortfolioTradingEnv for DRL-based stock trading.
Initial Random Policy Evaluation

Evaluates a random trading strategy to measure baseline performance.
Trains the DDPG Model in Stages

Trains in 5 incremental stages (each with 20% of total timesteps).
Saves intermediate checkpoints at each stage.
Conducts quick evaluations between stages.
Final Model Evaluation & Saving

Saves the fully trained DDPG model.
Prints training completion time and path to saved model.
Ideal Use Case:
This function is suitable for algorithmic trading firms, quantitative researchers, and hedge funds looking to train AI-driven portfolio management systems. It enables investors to automate trading strategies, maximizing returns while minimizing risk exposure.


In [7]:


def train_model(data_path, model_save_path, max_stocks=100, lookback=30, timesteps=500000):
    print(f"Starting training at {datetime.now()}")

    # Initialize components with more manageable parameters
    train_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=20
    )

    def make_env():
        return PortfolioTradingEnv(train_iter)

    train_env = DummyVecEnv([make_env])

    # Configure DDPG model with suitable hyperparameters
    n_actions = train_iter.num_stocks + 1  # Action space size
    action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

    model = DDPG(
        "MlpPolicy",
        train_env,
        learning_rate=1e-3,
        buffer_size=1000000,
        batch_size=128,
        tau=0.005,  # Soft update factor
        gamma=0.99,
        train_freq=(1, "episode"),
        gradient_steps=-1,
        action_noise=action_noise,
        verbose=1,
        device="auto",  # Use GPU if available
        tensorboard_log="./ddpg_portfolio_tensorboard/"
    )

    # Log initial random policy performance
    print("Evaluating random policy before training...")
    try:
        obs = train_env.reset()
        done = np.array([False])
        total_reward = 0
        step_count = 0
        max_steps = 100  # Limit evaluation to avoid infinite loops

        while not done.any() and step_count < max_steps:
            actions = []
            for i in range(train_env.num_envs):
                action = np.random.random(train_iter.num_stocks + 1)
                action = action / action.sum()
                actions.append(action)

            actions = np.array(actions)
            obs, rewards, done, info = train_env.step(actions)
            total_reward += rewards[0]
            step_count += 1

        print(f"Random policy evaluation: Total steps: {step_count}, Total reward: {total_reward}")
    except Exception as e:
        print(f"Error during random policy evaluation: {e}")
        print("Continuing with training anyway...")

    # Start training
    print(f"Training model with {train_iter.num_stocks} stocks, each with {lookback} days of history")

    stage_timesteps = timesteps // 5
    for stage in range(5):
        print(f"\nTraining stage {stage+1}/5 ({stage_timesteps} timesteps)...")
        try:
            model.learn(total_timesteps=stage_timesteps, progress_bar=True, reset_num_timesteps=False)

            # Save checkpoint
            checkpoint_path = f"{model_save_path}_checkpoint_{stage+1}"
            model.save(checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")

            # Quick evaluation
            if stage < 4:  # Skip final evaluation as we'll do a full one later
                print("Quick evaluation of current policy...")
                eval_env = make_env()
                for i in range(3):
                    obs, _ = eval_env.reset()
                    done = False
                    episode_reward = 0
                    step_count = 0
                    max_eval_steps = 100

                    while not done and step_count < max_eval_steps:
                        action, _ = model.predict(obs, deterministic=True)
                        obs, reward, done, _, info = eval_env.step(action)
                        episode_reward += reward
                        step_count += 1

                    print(f"  Episode {i+1} reward: {episode_reward:.4f}, Steps: {step_count}, "
                          f"Final portfolio: ${info['portfolio_value']:.2f}")
        except Exception as e:
            print(f"Error during training stage {stage+1}: {e}")
            print("Attempting to continue with next stage...")

    # Save the final trained model
    try:
        model.save(model_save_path)
        print(f"Training completed at {datetime.now()} and model saved to {model_save_path}!")
    except Exception as e:
        print(f"Error saving model: {e}")
        print("Training completed but model could not be saved.")

    return model

# Calculate common portfolio performance metrics

Portfolio Performance Metrics Calculator
Description:
The calculate_portfolio_metrics function computes essential financial performance metrics to evaluate the profitability and risk of a traded stock portfolio. It helps investors and AI trading models assess returns, volatility, and risk-adjusted performance.

This function is particularly useful for analyzing Deep Reinforcement Learning (DRL)-based trading strategies, as it quantifies how well an AI portfolio optimizer performs over time.

Key Metrics:
Total Return (%):

Measures the overall portfolio growth from start to end.
Formula:
Total Return
=(Final Portfolio Value/Initial Portfolio Value)−1



Annualized Return (%):

Adjusts daily returns to estimate yearly growth.
Assumes 252 trading days per year.
Annualized Volatility (%):

Measures risk exposure using standard deviation of returns.
High volatility indicates higher risk.
Sharpe Ratio:

Risk-adjusted return metric that compares the excess return (above the risk-free rate) to portfolio volatility.
Formula:
Sharpe Ratio
=(Annualized Return−Risk-Free Rate)/Annualized Volatility

A higher Sharpe ratio suggests better risk-adjusted performance.
Maximum Drawdown (%):

Measures worst portfolio loss from peak to trough.
Useful for assessing downside risk.
Win Rate (%):

Percentage of positive return days over the total period.
How It Works:
Validates Input:

Ensures that at least two portfolio values exist for meaningful analysis.
Computes Returns & Risk Metrics:

Daily returns are calculated using log returns.
Annualizes returns & volatility using 252 trading days assumption.
Calculates Maximum Drawdown:

Tracks highest portfolio value and worst loss from peak.
Returns Performance Summary:

Outputs key portfolio performance indicators in a structured format.
Ideal Use Case:
This function is essential for quantitative traders, algorithmic trading teams, and hedge funds looking to analyze investment strategies. It provides insights into risk-adjusted returns, helping investors optimize AI-driven portfolio management systems.

In [8]:
def calculate_portfolio_metrics(portfolio_values, risk_free_rate=0.02):
    """Calculate common portfolio performance metrics"""
    if len(portfolio_values) < 2:
        return {
            "total_return": 0,
            "sharpe_ratio": 0,
            "max_drawdown": 0,
            "volatility": 0
        }

    # Calculate returns
    returns = np.array([(portfolio_values[i] / portfolio_values[i-1]) - 1
                       for i in range(1, len(portfolio_values))])

    # Calculate metrics
    total_return = (portfolio_values[-1] / portfolio_values[0]) - 1
    daily_returns_mean = np.mean(returns)
    daily_returns_std = np.std(returns)

    # Annualize (assuming 252 trading days)
    annualized_return = ((1 + daily_returns_mean) ** 252) - 1
    annualized_vol = daily_returns_std * np.sqrt(252)

    # Sharpe ratio
    sharpe_ratio = (annualized_return - risk_free_rate) / annualized_vol if annualized_vol > 0 else 0

    # Maximum drawdown
    peak = portfolio_values[0]
    max_drawdown = 0

    for value in portfolio_values:
        if value > peak:
            peak = value
        drawdown = (peak - value) / peak
        max_drawdown = max(max_drawdown, drawdown)

    return {
        "total_return": total_return * 100,  # Convert to percentage
        "annualized_return": annualized_return * 100,
        "volatility": annualized_vol * 100,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown": max_drawdown * 100,
        "win_rate": np.mean(returns > 0) * 100
    }



# Backtesting Portfolio Strategy Using Deep Reinforcement Learning

Backtesting Portfolio Strategy Using Deep Reinforcement Learning
Description:
The backtest_portfolio function simulates a historical trading strategy using a trained Deep Reinforcement Learning (DRL) model. It evaluates how well the AI-based portfolio optimizer performs over a given backtest period, tracking key financial metrics like portfolio value, returns, transaction costs, diversification, and cash allocation.

This function helps traders and investors assess the real-world performance of an AI-powered portfolio management strategy before live deployment.

Key Features:
Simulated Trading Environment:
Uses a StockPortfolioIterator to process historical stock data.
Runs the strategy in a PortfolioTradingEnv, mimicking real trading conditions.
Performance Metrics Tracking:
Portfolio Value Over Time → Tracks capital growth.
Returns (%) → Measures daily portfolio returns.
Cash Allocations → Monitors cash vs. equity allocation.
Transaction Costs → Tracks fees incurred from trading.
Holdings Diversification → Measures the number of stocks in the portfolio with significant allocation (>1%).
Daily Turnover → Quantifies portfolio rebalancing activity.
Dynamic Strategy Execution:
Runs step-by-step trading decisions using the AI model.
Records portfolio performance after each trade.
Incremental Progress Logging:
Displays real-time progress every 50 trading step



In [9]:
def backtest_portfolio(model, data_path, max_stocks=100, lookback=30, initial_cash=10000, years=1):
    """Run a comprehensive backtest of the portfolio strategy"""
    print(f"Running {years}-year backtest simulation...")

    # Initialize the environment with the backtest data
    backtest_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=years
    )

    backtest_env = PortfolioTradingEnv(backtest_iter, initial_cash=initial_cash)

    # Run backtest
    obs, _ = backtest_env.reset()
    done = False

    portfolio_values = [initial_cash]
    returns = []
    cash_allocations = []
    transaction_costs = []
    holdings_diversification = []
    daily_turnover = []

    step = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        # Track portfolio composition
        cash_allocations.append(action[0])

        # Number of stocks with allocation > 1%
        significant_holdings = sum(1 for a in action[1:] if a > 0.01)
        holdings_diversification.append(significant_holdings)

        # Execute step
        obs, reward, done, _, info = backtest_env.step(action)

        # Track metrics
        portfolio_values.append(info["portfolio_value"])
        if step > 0:
            returns.append((portfolio_values[-1] / portfolio_values[-2]) - 1)
        transaction_costs.append(info["transaction_costs"])

        # Calculate turnover (sum of absolute changes in allocation)
        if step == 0:
            daily_turnover.append(0)
        else:
            turnover = info["transaction_costs"] / (portfolio_values[-2] * backtest_env.transaction_cost)
            daily_turnover.append(turnover)

        step += 1

        # Print progress
        if step % 50 == 0:
            print(f"Backtest step {step}, Portfolio value: ${portfolio_values[-1]:.2f}")

    # Calculate performance metrics
    metrics = calculate_portfolio_metrics(portfolio_values)

    backtest_results = {
        "portfolio_values": portfolio_values,
        "returns": returns,
        "cash_allocations": cash_allocations,
        "transaction_costs": transaction_costs,
        "holdings_diversification": holdings_diversification,
        "daily_turnover": daily_turnover,
        "metrics": metrics,
        "steps": step,
        "final_value": portfolio_values[-1]
    }

    return backtest_results

In [10]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Portfolio Optimization Main Execution Script
Description:
The __main__ script serves as the entry point for running the Deep Reinforcement Learning (DRL) portfolio optimization system. It provides a command-line interface (CLI) and Jupyter Notebook compatibility for training, evaluating, backtesting, and generating investment recommendations using a trained DDPG model.

This script ensures seamless execution by supporting different modes:

Train a new model
Evaluate an existing model
Backtest a strategy
Generate a portfolio recommendation
Train and evaluate together
Key Features:
Flexible Execution Modes:
train → Trains a new DDPG-based AI trading model.
evaluate → Assesses a pre-trained model on validation data.
backtest → Simulates the model’s trading decisions on historical stock data.
recommend → Provides an AI-driven stock allocation strategy.
train_and_evaluate → Trains and evaluates in one run.
Automatic Parameter Selection:
Uses default settings in Jupyter/Colab for convenience.
Supports command-line argument parsing for terminal execution.
Incremental Model Training with Checkpoints:
Saves intermediate training progress to avoid data loss.
Comprehensive Evaluation & Metrics Tracking:
Computes risk-adjusted return metrics (Sharpe Ratio, Volatility, Max Drawdown, etc.).
Automated JSON Logging for Results:
Saves evaluation, backtest, and recommendation results in the results/ directory.

In [17]:
if __name__ == "__main__":
    import sys
    import os
    import json
    from datetime import datetime

    # Default parameters
    DATA_PATH = "/content/drive/MyDrive/historical_data.csv"
    MODEL_SAVE_PATH = "/content/drive/MyDrive/ddpg_portfolio_model"
    MAX_STOCKS = 100  # Limit number of stocks to make training feasible
    LOOKBACK = 30     # Shorter lookback period
    TIMESTEPS = 20000  # Increased number of training steps
    INITIAL_CASH = 10000
    MODE = "train_and_evaluate"  # Default mode

    # Check if running in Jupyter/Colab environment
    is_notebook = 'ipykernel' in sys.modules

    if is_notebook:
        # For Jupyter/Colab, don't use argparse
        # You can modify these values directly in the notebook
        print("Running in notebook environment. Using default parameters.")
        print(f"DATA_PATH: {DATA_PATH}")
        print(f"MODEL_SAVE_PATH: {MODEL_SAVE_PATH}")
        print(f"MAX_STOCKS: {MAX_STOCKS}")
        print(f"LOOKBACK: {LOOKBACK}")
        print(f"TIMESTEPS: {TIMESTEPS}")
        print(f"INITIAL_CASH: {INITIAL_CASH}")
        print(f"MODE: {MODE}")

        # If you want to change parameters, do it here
        # Example: MODE = "recommend"
    else:
        # For command line execution, use argparse
        import argparse
        parser = argparse.ArgumentParser(description="Portfolio Optimization with Reinforcement Learning")
        parser.add_argument("--data_path", type=str, default=DATA_PATH, help="Path to historical data CSV")
        parser.add_argument("--model_path", type=str, default=MODEL_SAVE_PATH, help="Path to save/load model")
        parser.add_argument("--max_stocks", type=int, default=MAX_STOCKS, help="Maximum number of stocks to consider")
        parser.add_argument("--lookback", type=int, default=LOOKBACK, help="Lookback window size")
        parser.add_argument("--timesteps", type=int, default=TIMESTEPS, help="Number of training timesteps")
        parser.add_argument("--initial_cash", type=float, default=INITIAL_CASH, help="Initial cash for portfolio")
        parser.add_argument("--mode", type=str, default=MODE,
                            choices=["train", "evaluate", "backtest", "recommend", "train_and_evaluate"],
                            help="Operation mode")

        args = parser.parse_args()

        # Update variables from command line args
        DATA_PATH = args.data_path
        MODEL_SAVE_PATH = args.model_path
        MAX_STOCKS = args.max_stocks
        LOOKBACK = args.lookback
        TIMESTEPS = args.timesteps
        INITIAL_CASH = args.initial_cash
        MODE = args.mode

    # Create results directory
    os.makedirs("results", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Main execution logic
    if MODE == "train" or MODE == "train_and_evaluate":
        print(f"\n{'='*80}\nSTARTING TRAINING\n{'='*80}")
        model = train_model(
            DATA_PATH,
            MODEL_SAVE_PATH,
            MAX_STOCKS,
            LOOKBACK,
            TIMESTEPS
        )
        print(f"Model saved to {MODEL_SAVE_PATH}")
    else:
        # Load existing model
        try:
            from stable_baselines3 import DDPG
            print(f"Loading model from {MODEL_SAVE_PATH}")
            model = DDPG.load(MODEL_SAVE_PATH)
        except Exception as e:
            print(f"Error loading model: {e}")
            print("Please train a model first or provide a valid model path.")
            exit(1)

    if MODE == "evaluate" or MODE == "train_and_evaluate":
        print(f"\n{'='*80}\nEVALUATING MODEL\n{'='*80}")
        # Evaluate the model
        rewards, values = evaluate_model(
            model,
            DATA_PATH,
            MAX_STOCKS,
            LOOKBACK,
            episodes=10
        )

        # Save evaluation results
        eval_results = {
            "rewards": [float(r) for r in rewards],
            "final_values": [float(v) for v in values],
            "avg_reward": float(np.mean(rewards)),
            "avg_final_value": float(np.mean(values)),
            "max_reward": float(np.max(rewards)),
            "min_reward": float(np.min(rewards)),
            "success_rate": float(np.mean(np.array(rewards) > 0) * 100)
        }

        with open(f"results/evaluation_{timestamp}.json", "w") as f:
            json.dump(eval_results, f, indent=2)

        print(f"Evaluation results saved to results/evaluation_{timestamp}.json")

    if MODE == "backtest":
        print(f"\n{'='*80}\nRUNNING BACKTEST\n{'='*80}")
        # Run comprehensive backtest
        backtest_results = backtest_portfolio(
            model,
            DATA_PATH,
            MAX_STOCKS,
            LOOKBACK,
            INITIAL_CASH,
            years=2  # Use 2 years of data for backtest
        )

        # Print backtest summary
        metrics = backtest_results["metrics"]
        print("\nBacktest Summary:")
        print(f"Total Return: {metrics['total_return']:.2f}%")
        print(f"Annualized Return: {metrics['annualized_return']:.2f}%")
        print(f"Volatility: {metrics['volatility']:.2f}%")
        print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
        print(f"Maximum Drawdown: {metrics['max_drawdown']:.2f}%")
        print(f"Win Rate: {metrics['win_rate']:.2f}%")
        print(f"Average Cash Allocation: {np.mean(backtest_results['cash_allocations']) * 100:.2f}%")
        print(f"Average Number of Holdings: {np.mean(backtest_results['holdings_diversification']):.1f} stocks")
        print(f"Average Daily Turnover: {np.mean(backtest_results['daily_turnover']) * 100:.2f}%")

        # Save backtest results (excluding large arrays)
        backtest_summary = {
            "initial_cash": INITIAL_CASH,
            "final_value": float(backtest_results["final_value"]),
            "trading_days": backtest_results["steps"],
            "metrics": {k: float(v) for k, v in metrics.items()},
            "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
            "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
            "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100),
            "avg_transaction_costs": float(np.mean(backtest_results["transaction_costs"]))
        }

        with open(f"results/backtest_{timestamp}.json", "w") as f:
            json.dump(backtest_summary, f, indent=2)

        print(f"Backtest summary saved to results/backtest_{timestamp}.json")

    if MODE == "recommend":
        print(f"\n{'='*80}\nGENERATING PORTFOLIO RECOMMENDATION\n{'='*80}")
        # Generate portfolio recommendation
        recommender = PortfolioRecommender(
            MODEL_SAVE_PATH,
            DATA_PATH,
            max_stocks=MAX_STOCKS,
            lookback=LOOKBACK
        )

        recommendation = recommender.recommend_portfolio(INITIAL_CASH)

        # Print recommendation
        print(f"Recommended portfolio allocation (as of {recommendation['recommendation_date']}):")
        print(f"Cash: ${recommendation['cash_amount']:.2f} ({recommendation['cash_percent']:.2f}%)")
        print("\nStock allocations:")

        # Sort allocations by percentage (largest first)
        sorted_allocations = sorted(
            recommendation['stock_allocations'].items(),
            key=lambda x: x[1]['allocation_percent'],
            reverse=True
        )

        for ticker, details in sorted_allocations:
            if details['allocation_percent'] > 1.0:  # Only show significant allocations
                print(f"{ticker}: ${details['allocation_cad']:.2f} "
                      f"({details['allocation_percent']:.2f}%) - "
                      f"{details['shares']:.4f} shares @ ${details['price_per_share']:.2f}")

        # Calculate some portfolio statistics
        if sorted_allocations:
            allocations = [details['allocation_percent'] for _, details in sorted_allocations]
            print("\nPortfolio allocation statistics:")
            print(f"Number of stocks: {len(allocations)}")
            print(f"Max allocation: {max(allocations):.2f}%")
            print(f"Min allocation: {min(allocations):.2f}%")
            print(f"Avg allocation: {sum(allocations)/len(allocations):.2f}%")
            print(f"Diversification score: {len([a for a in allocations if a > 1.0])}")

        # Save recommendation
        with open(f"results/recommendation_{timestamp}.json", "w") as f:
            json.dump(recommendation, f, indent=2)

        print(f"Recommendation saved to results/recommendation_{timestamp}.json")

    print(f"\n{'='*80}\nPORTFOLIO OPTIMIZATION COMPLETE\n{'='*80}")
    print(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"All results saved in the 'results' directory.")

Running in notebook environment. Using default parameters.
DATA_PATH: /content/drive/MyDrive/historical_data.csv
MODEL_SAVE_PATH: /content/drive/MyDrive/ddpg_portfolio_model
MAX_STOCKS: 100
LOOKBACK: 30
TIMESTEPS: 20000
INITIAL_CASH: 10000
MODE: train_and_evaluate

STARTING TRAINING
Starting training at 2025-04-01 01:20:18.879154
Loading data from /content/drive/MyDrive/historical_data.csv...
Adding technical indicators...
Technical indicators added successfully
Scaling features...
Preparing training data...
Loaded 100 stocks with 5021 trading days
Date range: 2004-12-30 00:00:00 to 2024-12-30 00:00:00
Using cuda device


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 168.41GB > 11.06GB
  warnings.warn(


Evaluating random policy before training...
Step 100: Reward = 0.001895, Portfolio Value: 9548.05, Transaction Costs: 7.17
Random policy evaluation: Total steps: 100, Total reward: -0.12438686937093735
Training model with 100 stocks, each with 30 days of history

Training stage 1/5 (4000 timesteps)...
Logging to ./ddpg_portfolio_tensorboard/DDPG_0


Output()

/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

...

Step 4600: Reward = 0.028111, Portfolio Value: 33511.96, Transaction Costs: 2.38

Step 4700: Reward = 0.016564, Portfolio Value: 39227.14, Transaction Costs: 2.11

Step 4800: Reward = -0.003102, Portfolio Value: 40810.37, Transaction Costs: 2.36

Step 4891: Reward = -0.000139, Portfolio Value: 44666.05, Transaction Costs: 3.11

Checkpoint saved to /content/drive/MyDrive/ddpg_portfolio_model_checkpoint_5
Training completed at 2025-04-01 01:32:12.277866 and model saved to /content/drive/MyDrive/ddpg_portfolio_model!
Model saved to /content/drive/MyDrive/ddpg_portfolio_model

EVALUATING MODEL
Evaluating model performance...
Loading data from /content/drive/MyDrive/historical_data.csv...
Adding technical indicators...
Technical indicators added successfully
Scaling features...
Preparing training data...
Loaded 100 stocks with 1256 trading days
Date range: 2019-12-30 00:00:00 to 2024-12-30 00:00:00
Step 100: Reward = 0.008519, Portfolio Value: 9504.12, Transaction Costs: 0.28
Step 200: Reward = 0.003600, Portfolio Value: 12471.27, Transaction Costs: 0.52
Step 300: Reward = -0.010402, Portfolio Value: 17906.76, Transaction Costs: 0.58
Step 400: Reward = 0.016634, Portfolio Value: 24502.66, Transaction Costs: 0.75
Step 500: Reward = -0.007159, Portfolio Value: 26122.47, Transaction Costs: 1.13
Step 600: Reward = 0.0

# Portfolio Recommendation Execution
Description:
This script sets the mode to "recommend", instructing the AI system to generate an optimal stock portfolio allocation based on the trained DDPG reinforcement learning model. It analyzes recent market data and produces cash and stock allocations for a given investment amount.

Key Features:
Portfolio Allocation Strategy:
Cash Allocation → Determines how much capital remains in cash.
Stock Allocation → Distributes investment across highly liquid stocks.
Sorting & Filtering:
Displays only significant stock allocations (>1% allocation).
Sorts stocks by allocation percentage (highest to lowest).
Portfolio Statistics:
Number of stocks in the recommended portfolio.
Max, Min, and Average allocation per stock.
Diversification score (number of stocks with >1% allocation).
Automated Result Saving:
Saves recommendation output in the results/ directory as a JSON file.

In [18]:
# Change the mode to 'recommend'
MODE = "recommend"

# Optionally, you can modify the initial cash amount
INITIAL_CASH = 10000  # or any other amount you prefer

# Run the recommendation part of the code
if MODE == "recommend":
    print(f"\n{'='*80}\nGENERATING PORTFOLIO RECOMMENDATION\n{'='*80}")
    # Generate portfolio recommendation
    recommender = PortfolioRecommender(
        MODEL_SAVE_PATH,
        DATA_PATH,
        max_stocks=MAX_STOCKS,
        lookback=LOOKBACK
    )

    recommendation = recommender.recommend_portfolio(INITIAL_CASH)

    # Print recommendation
    print(f"Recommended portfolio allocation (as of {recommendation['recommendation_date']}):")
    print(f"Cash: ${recommendation['cash_amount']:.2f} ({recommendation['cash_percent']:.2f}%)")
    print("\nStock allocations:")

    # Sort allocations by percentage (largest first)
    sorted_allocations = sorted(
        recommendation['stock_allocations'].items(),
        key=lambda x: x[1]['allocation_percent'],
        reverse=True
    )

    for ticker, details in sorted_allocations:
        if details['allocation_percent'] > 1.0:  # Only show significant allocations
            print(f"{ticker}: ${details['allocation_cad']:.2f} "
                  f"({details['allocation_percent']:.2f}%) - "
                  f"{details['shares']:.4f} shares @ ${details['price_per_share']:.2f}")

    # Calculate some portfolio statistics
    if sorted_allocations:
        allocations = [details['allocation_percent'] for _, details in sorted_allocations]
        print("\nPortfolio allocation statistics:")
        print(f"Number of stocks: {len(allocations)}")
        print(f"Max allocation: {max(allocations):.2f}%")
        print(f"Min allocation: {min(allocations):.2f}%")
        print(f"Avg allocation: {sum(allocations)/len(allocations):.2f}%")
        print(f"Diversification score: {len([a for a in allocations if a > 1.0])}")

    # Save recommendation
    with open(f"results/recommendation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json", "w") as f:
        import json
        json.dump(recommendation, f, indent=2)

    print(f"Recommendation saved to results directory")


GENERATING PORTFOLIO RECOMMENDATION


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 168.41GB > 6.39GB
  warnings.warn(


Recommended portfolio allocation (as of 2024-12-30 00:00:00):
Cash: $212.77 (2.13%)

Stock allocations:
AEM.TO: $212.77 (2.13%) - 1.9033 shares @ $111.79
ATD.TO: $212.77 (2.13%) - 2.6888 shares @ $79.13
BB.TO: $212.77 (2.13%) - 38.6145 shares @ $5.51
BITF.TO: $212.77 (2.13%) - 95.4107 shares @ $2.23
BLDP.TO: $212.77 (2.13%) - 88.6525 shares @ $2.40
BMO.TO: $212.77 (2.13%) - 1.5294 shares @ $139.12
BTCC.TO: $212.77 (2.13%) - 12.1720 shares @ $17.48
BTE.TO: $212.77 (2.13%) - 60.4449 shares @ $3.52
BTO.TO: $212.77 (2.13%) - 61.3158 shares @ $3.47
CM.TO: $212.77 (2.13%) - 2.3358 shares @ $91.09
CNQ.TO: $212.77 (2.13%) - 4.8991 shares @ $43.43
CNR.TO: $212.77 (2.13%) - 1.4698 shares @ $144.76
DLR.TO: $212.77 (2.13%) - 14.6241 shares @ $14.55
DML.TO: $212.77 (2.13%) - 79.9872 shares @ $2.66
ENB.TO: $212.77 (2.13%) - 3.5180 shares @ $60.48
FRU.TO: $212.77 (2.13%) - 16.8728 shares @ $12.61
FTS.TO: $212.77 (2.13%) - 3.5556 shares @ $59.84
GEI.TO: $212.77 (2.13%) - 8.7092 shares @ $24.43
GLO.TO:

# Portfolio Backtesting Execution
Description:
This script sets the mode to "backtest", instructing the AI trading system to simulate a trading strategy over a historical period (2 years) using a trained DDPG model. It evaluates how well the AI-driven portfolio optimization model performs in realistic market conditions.

Key Features:
Simulated Trading Environment:
Uses real historical stock data to test portfolio performance.
Allocates cash and stock holdings dynamically based on AI model predictions.
Performance Metrics Tracking:
Total Return (%) → Measures overall portfolio growth.
Annualized Return (%) → Estimates expected yearly return.
Volatility (%) → Measures risk exposure.
Sharpe Ratio → Evaluates risk-adjusted return.
Maximum Drawdown (%) → Assesses worst peak-to-trough loss.
Win Rate (%) → Tracks profitable trading periods.
Portfolio Behavior Analysis:
Cash Allocation (%) → Tracks how much capital is held in cash.
Holdings Diversification → Counts number of significant stock positions.
Daily Turnover (%) → Monitors portfolio rebalancing activity.
Automated Logging & JSON Export:
Saves backtest results to a timestamped JSON file in the results/ directory.


In [19]:
# Change the mode to 'backtest'
MODE = "backtest"

# Optionally, customize these parameters
INITIAL_CASH = 10000  # Starting portfolio value
LOOKBACK = 30         # Days of history to consider
MAX_STOCKS = 100      # Maximum number of stocks to consider

# Run the backtest
if MODE == "backtest":
    print(f"\n{'='*80}\nRUNNING BACKTEST\n{'='*80}")
    # Run comprehensive backtest
    backtest_results = backtest_portfolio(
        model,
        DATA_PATH,
        MAX_STOCKS,
        LOOKBACK,
        INITIAL_CASH,
        years=2  # Use 2 years of data for backtest
    )

    # Print backtest summary
    metrics = backtest_results["metrics"]
    print("\nBacktest Summary:")
    print(f"Total Return: {metrics['total_return']:.2f}%")
    print(f"Annualized Return: {metrics['annualized_return']:.2f}%")
    print(f"Volatility: {metrics['volatility']:.2f}%")
    print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
    print(f"Maximum Drawdown: {metrics['max_drawdown']:.2f}%")
    print(f"Win Rate: {metrics['win_rate']:.2f}%")
    print(f"Average Cash Allocation: {np.mean(backtest_results['cash_allocations']) * 100:.2f}%")
    print(f"Average Number of Holdings: {np.mean(backtest_results['holdings_diversification']):.1f} stocks")
    print(f"Average Daily Turnover: {np.mean(backtest_results['daily_turnover']) * 100:.2f}%")

    # Save backtest results (excluding large arrays)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backtest_summary = {
        "initial_cash": INITIAL_CASH,
        "final_value": float(backtest_results["final_value"]),
        "trading_days": backtest_results["steps"],
        "metrics": {k: float(v) for k, v in metrics.items()},
        "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
        "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
        "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100),
        "avg_transaction_costs": float(np.mean(backtest_results["transaction_costs"]))
    }

    with open(f"results/backtest_{timestamp}.json", "w") as f:
        import json
        json.dump(backtest_summary, f, indent=2)

    print(f"Backtest summary saved to results/backtest_{timestamp}.json")


RUNNING BACKTEST
Running 2-year backtest simulation...
Loading data from /content/drive/MyDrive/historical_data.csv...
Adding technical indicators...
Technical indicators added successfully
Scaling features...
Preparing training data...
Loaded 100 stocks with 502 trading days
Date range: 2022-12-30 00:00:00 to 2024-12-30 00:00:00
Backtest step 50, Portfolio value: $9871.36
Step 100: Reward = 0.014201, Portfolio Value: 9674.73, Transaction Costs: 0.57
Backtest step 100, Portfolio value: $9674.73
Backtest step 150, Portfolio value: $10201.11
Step 200: Reward = -0.000849, Portfolio Value: 10009.70, Transaction Costs: 1.10
Backtest step 200, Portfolio value: $10009.70
Backtest step 250, Portfolio value: $10253.41
Step 300: Reward = -0.001105, Portfolio Value: 11284.03, Transaction Costs: 0.56
Backtest step 300, Portfolio value: $11284.03
Backtest step 350, Portfolio value: $11348.36
Step 400: Reward = 0.007139, Portfolio Value: 11320.03, Transaction Costs: 0.53
Backtest step 400, Portfoli

In [20]:
# Change the mode to 'recommend'
MODE = "recommend"

# Optionally, you can modify the initial cash amount
INITIAL_CASH = 20000  # or any other amount you prefer

# Run the recommendation part of the code
if MODE == "recommend":
    print(f"\n{'='*80}\nGENERATING PORTFOLIO RECOMMENDATION\n{'='*80}")
    # Generate portfolio recommendation
    recommender = PortfolioRecommender(
        MODEL_SAVE_PATH,
        DATA_PATH,
        max_stocks=MAX_STOCKS,
        lookback=LOOKBACK
    )

    recommendation = recommender.recommend_portfolio(INITIAL_CASH)

    # Print recommendation
    print(f"Recommended portfolio allocation (as of {recommendation['recommendation_date']}):")
    print(f"Cash: ${recommendation['cash_amount']:.2f} ({recommendation['cash_percent']:.2f}%)")
    print("\nStock allocations:")

    # Sort allocations by percentage (largest first)
    sorted_allocations = sorted(
        recommendation['stock_allocations'].items(),
        key=lambda x: x[1]['allocation_percent'],
        reverse=True
    )

    for ticker, details in sorted_allocations:
        if details['allocation_percent'] > 1.0:  # Only show significant allocations
            print(f"{ticker}: ${details['allocation_cad']:.2f} "
                  f"({details['allocation_percent']:.2f}%) - "
                  f"{details['shares']:.4f} shares @ ${details['price_per_share']:.2f}")

    # Calculate some portfolio statistics
    if sorted_allocations:
        allocations = [details['allocation_percent'] for _, details in sorted_allocations]
        print("\nPortfolio allocation statistics:")
        print(f"Number of stocks: {len(allocations)}")
        print(f"Max allocation: {max(allocations):.2f}%")
        print(f"Min allocation: {min(allocations):.2f}%")
        print(f"Avg allocation: {sum(allocations)/len(allocations):.2f}%")
        print(f"Diversification score: {len([a for a in allocations if a > 1.0])}")

    # Save recommendation
    with open(f"results/recommendation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json", "w") as f:
        import json
        json.dump(recommendation, f, indent=2)

    print(f"Recommendation saved to results directory")


GENERATING PORTFOLIO RECOMMENDATION


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 168.41GB > 6.40GB
  warnings.warn(


Recommended portfolio allocation (as of 2024-12-30 00:00:00):
Cash: $425.53 (2.13%)

Stock allocations:
AEM.TO: $425.53 (2.13%) - 3.8065 shares @ $111.79
ATD.TO: $425.53 (2.13%) - 5.3776 shares @ $79.13
BB.TO: $425.53 (2.13%) - 77.2290 shares @ $5.51
BITF.TO: $425.53 (2.13%) - 190.8215 shares @ $2.23
BLDP.TO: $425.53 (2.13%) - 177.3049 shares @ $2.40
BMO.TO: $425.53 (2.13%) - 3.0587 shares @ $139.12
BTCC.TO: $425.53 (2.13%) - 24.3439 shares @ $17.48
BTE.TO: $425.53 (2.13%) - 120.8897 shares @ $3.52
BTO.TO: $425.53 (2.13%) - 122.6317 shares @ $3.47
CM.TO: $425.53 (2.13%) - 4.6716 shares @ $91.09
CNQ.TO: $425.53 (2.13%) - 9.7981 shares @ $43.43
CNR.TO: $425.53 (2.13%) - 2.9396 shares @ $144.76
DLR.TO: $425.53 (2.13%) - 29.2482 shares @ $14.55
DML.TO: $425.53 (2.13%) - 159.9744 shares @ $2.66
ENB.TO: $425.53 (2.13%) - 7.0359 shares @ $60.48
FRU.TO: $425.53 (2.13%) - 33.7456 shares @ $12.61
FTS.TO: $425.53 (2.13%) - 7.1112 shares @ $59.84
GEI.TO: $425.53 (2.13%) - 17.4184 shares @ $24.43
G

In [21]:
# Change the mode to 'recommend'
MODE = "recommend"

# Optionally, you can modify the initial cash amount
INITIAL_CASH = 25000  # or any other amount you prefer

# Run the recommendation part of the code
if MODE == "recommend":
    print(f"\n{'='*80}\nGENERATING PORTFOLIO RECOMMENDATION\n{'='*80}")
    # Generate portfolio recommendation
    recommender = PortfolioRecommender(
        MODEL_SAVE_PATH,
        DATA_PATH,
        max_stocks=MAX_STOCKS,
        lookback=LOOKBACK
    )

    recommendation = recommender.recommend_portfolio(INITIAL_CASH)

    # Print recommendation
    print(f"Recommended portfolio allocation (as of {recommendation['recommendation_date']}):")
    print(f"Cash: ${recommendation['cash_amount']:.2f} ({recommendation['cash_percent']:.2f}%)")
    print("\nStock allocations:")

    # Sort allocations by percentage (largest first)
    sorted_allocations = sorted(
        recommendation['stock_allocations'].items(),
        key=lambda x: x[1]['allocation_percent'],
        reverse=True
    )

    for ticker, details in sorted_allocations:
        if details['allocation_percent'] > 1.0:  # Only show significant allocations
            print(f"{ticker}: ${details['allocation_cad']:.2f} "
                  f"({details['allocation_percent']:.2f}%) - "
                  f"{details['shares']:.4f} shares @ ${details['price_per_share']:.2f}")

    # Calculate some portfolio statistics
    if sorted_allocations:
        allocations = [details['allocation_percent'] for _, details in sorted_allocations]
        print("\nPortfolio allocation statistics:")
        print(f"Number of stocks: {len(allocations)}")
        print(f"Max allocation: {max(allocations):.2f}%")
        print(f"Min allocation: {min(allocations):.2f}%")
        print(f"Avg allocation: {sum(allocations)/len(allocations):.2f}%")
        print(f"Diversification score: {len([a for a in allocations if a > 1.0])}")

    # Save recommendation
    with open(f"results/recommendation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json", "w") as f:
        import json
        json.dump(recommendation, f, indent=2)

    print(f"Recommendation saved to results directory")


GENERATING PORTFOLIO RECOMMENDATION


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 168.41GB > 6.41GB
  warnings.warn(


Recommended portfolio allocation (as of 2024-12-30 00:00:00):
Cash: $531.91 (2.13%)

Stock allocations:
AEM.TO: $531.91 (2.13%) - 4.7582 shares @ $111.79
ATD.TO: $531.91 (2.13%) - 6.7220 shares @ $79.13
BB.TO: $531.91 (2.13%) - 96.5363 shares @ $5.51
BITF.TO: $531.91 (2.13%) - 238.5268 shares @ $2.23
BLDP.TO: $531.91 (2.13%) - 221.6312 shares @ $2.40
BMO.TO: $531.91 (2.13%) - 3.8234 shares @ $139.12
BTCC.TO: $531.91 (2.13%) - 30.4299 shares @ $17.48
BTE.TO: $531.91 (2.13%) - 151.1122 shares @ $3.52
BTO.TO: $531.91 (2.13%) - 153.2896 shares @ $3.47
CM.TO: $531.91 (2.13%) - 5.8394 shares @ $91.09
CNQ.TO: $531.91 (2.13%) - 12.2476 shares @ $43.43
CNR.TO: $531.91 (2.13%) - 3.6745 shares @ $144.76
DLR.TO: $531.91 (2.13%) - 36.5602 shares @ $14.55
DML.TO: $531.91 (2.13%) - 199.9680 shares @ $2.66
ENB.TO: $531.91 (2.13%) - 8.7949 shares @ $60.48
FRU.TO: $531.91 (2.13%) - 42.1820 shares @ $12.61
FTS.TO: $531.91 (2.13%) - 8.8890 shares @ $59.84
GEI.TO: $531.91 (2.13%) - 21.7730 shares @ $24.43


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
import os
import json
from stable_baselines3 import DDPG
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise
import itertools
import time
from tqdm import tqdm


def hyperparameter_tuning(
    data_path,
    base_model_path="tuning_models",
    max_stocks_options=[50, 100],
    lookback_options=[15, 30, 45],
    learning_rate_options=[1e-4, 3e-4, 1e-3],
    gamma_options=[0.95, 0.99],
    buffer_size_options=[10000, 100000],
    action_noise_options=["normal", "ou"],
    noise_std_options=[0.1, 0.2],
    batch_size_options=[64, 128],
    transaction_cost_options=[0.001, 0.002],
    tau_options=[0.001, 0.005],
    timesteps=100000,  # Reduced for tuning
    eval_episodes=3,   # Reduced for tuning
    n_eval_envs=2
):
    """
    Perform hyperparameter tuning for the portfolio optimization model using DDPG.

    Parameters:
    -----------
    data_path : str
        Path to the historical data CSV
    base_model_path : str
        Base path to save trained models
    max_stocks_options : list
        Different numbers of stocks to consider
    lookback_options : list
        Different lookback window sizes
    learning_rate_options : list
        Different learning rates for DDPG
    gamma_options : list
        Different discount factors
    buffer_size_options : list
        Different replay buffer sizes
    action_noise_options : list
        Different action noise types ("normal" or "ou")
    noise_std_options : list
        Different noise standard deviations
    batch_size_options : list
        Different batch sizes for training
    transaction_cost_options : list
        Different transaction cost values
    tau_options : list
        Different soft update coefficients
    timesteps : int
        Number of timesteps for each training run
    eval_episodes : int
        Number of episodes for evaluation
    n_eval_envs : int
        Number of environments for parallel evaluation

    Returns:
    --------
    dict
        Results of hyperparameter tuning
    """
    # Create results directory
    os.makedirs(os.path.join(base_model_path, "results"), exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    tuning_results_path = os.path.join(base_model_path, "results", f"tuning_results_{timestamp}.json")

    # Track all results
    all_results = []

    # Generate parameter grid (selecting key parameters to avoid combinatorial explosion)
    parameter_grid = []
    for max_stocks, lookback, lr, gamma, buffer_size, action_noise, noise_std in itertools.product(
        max_stocks_options,
        lookback_options,
        learning_rate_options,
        gamma_options,
        buffer_size_options,
        action_noise_options,
        noise_std_options
    ):
        # Fixed parameters for initial tuning round
        batch_size = batch_size_options[0]
        transaction_cost = transaction_cost_options[0]
        tau = tau_options[0]

        parameter_grid.append({
            "max_stocks": max_stocks,
            "lookback": lookback,
            "learning_rate": lr,
            "gamma": gamma,
            "buffer_size": buffer_size,
            "action_noise": action_noise,
            "noise_std": noise_std,
            "batch_size": batch_size,
            "transaction_cost": transaction_cost,
            "tau": tau
        })

    print(f"Tuning with {len(parameter_grid)} parameter combinations")

    # Run tuning
    for i, params in enumerate(tqdm(parameter_grid, desc="Hyperparameter tuning")):
        start_time = time.time()

        try:
            # Extract parameters
            max_stocks = params["max_stocks"]
            lookback = params["lookback"]
            learning_rate = params["learning_rate"]
            gamma = params["gamma"]
            buffer_size = params["buffer_size"]
            action_noise_type = params["action_noise"]
            noise_std = params["noise_std"]
            batch_size = params["batch_size"]
            transaction_cost = params["transaction_cost"]
            tau = params["tau"]

            model_name = f"ddpg_model_{i+1}_ms{max_stocks}_lb{lookback}_lr{learning_rate}_g{gamma}_ns{noise_std}"
            model_path = os.path.join(base_model_path, model_name)

            print(f"\nTraining model {i+1}/{len(parameter_grid)}: {model_name}")

            # Initialize data iterator
            train_iter = StockPortfolioIterator(
                data_path,
                lookback_window=lookback,
                min_history=100,
                max_stocks=max_stocks,
                train_years=10  # Reduced for faster tuning
            )

            # Create environment
            def make_env(transaction_cost=transaction_cost):
                return PortfolioTradingEnv(train_iter.reset(), transaction_cost=transaction_cost)

            train_env = DummyVecEnv([make_env])

            # Get action dimension from environment
            action_dim = train_env.action_space.shape[0]

            # Set up action noise
            if action_noise_type == "normal":
                action_noise = NormalActionNoise(
                    mean=np.zeros(action_dim),
                    sigma=noise_std * np.ones(action_dim)
                )
            else:  # Ornstein-Uhlenbeck noise
                action_noise = OrnsteinUhlenbeckActionNoise(
                    mean=np.zeros(action_dim),
                    sigma=noise_std * np.ones(action_dim)
                )

            # Create DDPG model
            model = DDPG(
                "MlpPolicy",
                train_env,
                learning_rate=learning_rate,
                buffer_size=buffer_size,
                batch_size=batch_size,
                tau=tau,
                gamma=gamma,
                action_noise=action_noise,
                verbose=0,
                device="auto"
            )

            # Train model with reduced timesteps for tuning
            model.learn(total_timesteps=timesteps, progress_bar=True)

            # Evaluate model
            rewards, values = evaluate_model(
                model,
                data_path,
                max_stocks,
                lookback,
                episodes=eval_episodes,
                verbose=False
            )

            # Run backtest
            backtest_results = backtest_portfolio(
                model,
                data_path,
                max_stocks,
                lookback,
                initial_cash=10000,
                years=1,  # Reduced for faster tuning
                verbose=False
            )

            # Calculate metrics
            train_time = time.time() - start_time
            avg_reward = np.mean(rewards)
            avg_final_value = np.mean(values)
            success_rate = np.mean(np.array(rewards) > 0) * 100

            # Extract key backtest metrics
            backtest_metrics = backtest_results["metrics"]

            # Store results
            result = {
                "model_name": model_name,
                "parameters": params,
                "training_time": train_time,
                "evaluation": {
                    "avg_reward": float(avg_reward),
                    "avg_final_value": float(avg_final_value),
                    "success_rate": float(success_rate)
                },
                "backtest": {
                    "total_return": float(backtest_metrics["total_return"]),
                    "annualized_return": float(backtest_metrics["annualized_return"]),
                    "volatility": float(backtest_metrics["volatility"]),
                    "sharpe_ratio": float(backtest_metrics["sharpe_ratio"]),
                    "max_drawdown": float(backtest_metrics["max_drawdown"]),
                    "win_rate": float(backtest_metrics["win_rate"])
                },
                "portfolio_characteristics": {
                    "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
                    "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
                    "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100)
                }
            }

            all_results.append(result)

            # Save model
            model.save(model_path)

            # Save interim results
            with open(tuning_results_path, "w") as f:
                json.dump(all_results, f, indent=2)

            print(f"Model {i+1} results:")
            print(f"  Avg reward: {avg_reward:.4f}")
            print(f"  Avg final value: ${avg_final_value:.2f}")
            print(f"  Success rate: {success_rate:.2f}%")
            print(f"  Sharpe ratio: {backtest_metrics['sharpe_ratio']:.2f}")
            print(f"  Annualized return: {backtest_metrics['annualized_return']:.2f}%")
            print(f"  Max drawdown: {backtest_metrics['max_drawdown']:.2f}%")

        except Exception as e:
            print(f"Error in parameter set {i+1}: {e}")
            all_results.append({
                "model_name": f"model_{i+1}",
                "parameters": params,
                "error": str(e)
            })

    # Find best model by different metrics
    valid_results = [r for r in all_results if "error" not in r]

    if valid_results:
        best_by_reward = max(valid_results, key=lambda x: x["evaluation"]["avg_reward"])
        best_by_sharpe = max(valid_results, key=lambda x: x["backtest"]["sharpe_ratio"])
        best_by_return = max(valid_results, key=lambda x: x["backtest"]["annualized_return"])

        best_models = {
            "best_by_reward": best_by_reward,
            "best_by_sharpe": best_by_sharpe,
            "best_by_return": best_by_return
        }

        # Save final results
        final_results = {
            "timestamp": timestamp,
            "all_results": all_results,
            "best_models": best_models
        }

        with open(tuning_results_path, "w") as f:
            json.dump(final_results, f, indent=2)

        print("\nHyperparameter tuning complete!")
        print(f"Results saved to: {tuning_results_path}")

        print("\nBest model by evaluation reward:")
        print_model_summary(best_by_reward)

        print("\nBest model by Sharpe ratio:")
        print_model_summary(best_by_sharpe)

        print("\nBest model by annualized return:")
        print_model_summary(best_by_return)

        return final_results
    else:
        print("No valid results found.")
        return {"timestamp": timestamp, "all_results": all_results}


def print_model_summary(model_result):
    """Print a summary of model performance"""
    params = model_result["parameters"]
    eval_metrics = model_result["evaluation"]
    backtest_metrics = model_result["backtest"]
    portfolio_metrics = model_result["portfolio_characteristics"]

    print(f"Model: {model_result['model_name']}")
    print(f"Parameters: max_stocks={params['max_stocks']}, lookback={params['lookback']}, "
          f"lr={params['learning_rate']}, gamma={params['gamma']}, noise={params['action_noise']}({params['noise_std']})")
    print(f"Evaluation: reward={eval_metrics['avg_reward']:.4f}, final_value=${eval_metrics['avg_final_value']:.2f}")
    print(f"Backtest: return={backtest_metrics['annualized_return']:.2f}%, "
          f"sharpe={backtest_metrics['sharpe_ratio']:.2f}, "
          f"drawdown={backtest_metrics['max_drawdown']:.2f}%")
    print(f"Portfolio: cash={portfolio_metrics['avg_cash_allocation']:.2f}%, "
          f"holdings={portfolio_metrics['avg_holdings']:.1f}, "
          f"turnover={portfolio_metrics['avg_turnover']:.2f}%")


def fine_tune_best_model(
    data_path,
    base_results_path,
    best_model_type="best_by_sharpe",  # Options: best_by_reward, best_by_sharpe, best_by_return
    timesteps=500000,  # Full training for fine-tuning
    eval_episodes=10
):
    """
    Fine-tune the best DDPG model from initial hyperparameter tuning

    Parameters:
    -----------
    data_path : str
        Path to the historical data CSV
    base_results_path : str
        Path to the tuning results JSON file
    best_model_type : str
        Which "best" model to fine-tune
    timesteps : int
        Number of timesteps for fine-tuning
    eval_episodes : int
        Number of episodes for evaluation

    Returns:
    --------
    dict
        Results of fine-tuning
    """
    # Load tuning results
    with open(base_results_path, "r") as f:
        tuning_results = json.load(f)

    best_model = tuning_results["best_models"][best_model_type]
    best_params = best_model["parameters"]

    print(f"Fine-tuning the {best_model_type} model:")
    print_model_summary(best_model)

    # Extract parameters
    max_stocks = best_params["max_stocks"]
    lookback = best_params["lookback"]
    learning_rate = best_params["learning_rate"]
    gamma = best_params["gamma"]
    buffer_size = best_params["buffer_size"]
    action_noise_type = best_params["action_noise"]
    noise_std = best_params["noise_std"]
    batch_size = best_params["batch_size"]
    transaction_cost = best_params["transaction_cost"]
    tau = best_params["tau"]

    # Create a timestamp for this fine-tuning run
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_name = f"finetuned_ddpg_{best_model_type}_{timestamp}"
    model_path = os.path.join(os.path.dirname(base_results_path), model_name)

    # Initialize data iterator with full data
    train_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=20  # Full training data
    )

    # Create environment
    def make_env(transaction_cost=transaction_cost):
        return PortfolioTradingEnv(train_iter.reset(), transaction_cost=transaction_cost)

    train_env = DummyVecEnv([make_env])

    # Get action dimension from environment
    action_dim = train_env.action_space.shape[0]

    # Set up action noise (with reduced standard deviation for fine-tuning)
    reduced_noise_std = noise_std * 0.5  # Reduce exploration during fine-tuning
    if action_noise_type == "normal":
        action_noise = NormalActionNoise(
            mean=np.zeros(action_dim),
            sigma=reduced_noise_std * np.ones(action_dim)
        )
    else:  # Ornstein-Uhlenbeck noise
        action_noise = OrnsteinUhlenbeckActionNoise(
            mean=np.zeros(action_dim),
            sigma=reduced_noise_std * np.ones(action_dim)
        )

    # Create model
    model = DDPG(
        "MlpPolicy",
        train_env,
        learning_rate=learning_rate,
        buffer_size=buffer_size,
        batch_size=batch_size,
        tau=tau,
        gamma=gamma,
        action_noise=action_noise,
        verbose=1,
        device="auto",
        tensorboard_log=os.path.join(os.path.dirname(base_results_path), "fine_tuning_tensorboard")
    )

    print(f"\nFine-tuning with {timesteps} timesteps...")

    # Train model with full timesteps
    model.learn(total_timesteps=timesteps, progress_bar=True)
    model.save(model_path)

    print(f"Model saved to {model_path}")

    # Evaluate model
    print("\nEvaluating fine-tuned model...")
    rewards, values = evaluate_model(
        model,
        data_path,
        max_stocks,
        lookback,
        episodes=eval_episodes
    )

    # Run backtest
    print("\nRunning backtest with fine-tuned model...")
    backtest_results = backtest_portfolio(
        model,
        data_path,
        max_stocks,
        lookback,
        initial_cash=10000,
        years=2  # Full backtest
    )

    # Calculate metrics
    avg_reward = np.mean(rewards)
    avg_final_value = np.mean(values)
    success_rate = np.mean(np.array(rewards) > 0) * 100

    # Extract key backtest metrics
    backtest_metrics = backtest_results["metrics"]

    # Store results
    fine_tuning_result = {
        "model_name": model_name,
        "model_path": model_path,
        "original_model": best_model["model_name"],
        "parameters": best_params,
        "evaluation": {
            "avg_reward": float(avg_reward),
            "avg_final_value": float(avg_final_value),
            "success_rate": float(success_rate)
        },
        "backtest": {
            "total_return": float(backtest_metrics["total_return"]),
            "annualized_return": float(backtest_metrics["annualized_return"]),
            "volatility": float(backtest_metrics["volatility"]),
            "sharpe_ratio": float(backtest_metrics["sharpe_ratio"]),
            "max_drawdown": float(backtest_metrics["max_drawdown"]),
            "win_rate": float(backtest_metrics["win_rate"])
        },
        "portfolio_characteristics": {
            "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
            "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
            "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100)
        }
    }

    # Save results
    results_path = os.path.join(os.path.dirname(base_results_path), f"fine_tuning_results_{timestamp}.json")
    with open(results_path, "w") as f:
        json.dump(fine_tuning_result, f, indent=2)

    print(f"\nFine-tuning results saved to: {results_path}")
    print("\nFine-tuned model performance:")
    print(f"  Avg reward: {avg_reward:.4f}")
    print(f"  Avg final value: ${avg_final_value:.2f}")
    print(f"  Success rate: {success_rate:.2f}%")
    print(f"  Sharpe ratio: {backtest_metrics['sharpe_ratio']:.2f}")
    print(f"  Annualized return: {backtest_metrics['annualized_return']:.2f}%")
    print(f"  Max drawdown: {backtest_metrics['max_drawdown']:.2f}%")

    return fine_tuning_result


# The evaluate_model and backtest_portfolio functions remain mostly unchanged
# Only the prediction part is slightly different for DDPG

def evaluate_model(model, data_path, max_stocks=100, lookback=30, episodes=10, verbose=True):
    """Evaluate the model on a validation dataset with option to silence output"""
    if verbose:
        print(f"Evaluating model performance...")

    eval_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=5  # Use more recent data for evaluation
    )

    eval_env = PortfolioTradingEnv(eval_iter)

    total_rewards = []
    portfolio_values = []

    for ep in range(episodes):
        obs, _ = eval_env.reset()
        done = False
        episode_reward = 0
        initial_value = eval_env.initial_cash
        step_count = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info = eval_env.step(action)
            episode_reward += reward
            step_count += 1

            if done:
                final_value = info["portfolio_value"]
                portfolio_values.append(final_value)

                # Calculate annualized return
                if step_count > 0:
                    days = step_count
                    annualized_return = ((final_value / initial_value) ** (252 / days) - 1) * 100
                else:
                    annualized_return = 0

                if verbose:
                    print(f"Episode {ep+1}: Return = {episode_reward:.4f}, "
                          f"Steps = {step_count}, "
                          f"Final Value = ${final_value:.2f}, "
                          f"Annualized Return = {annualized_return:.2f}%")

        total_rewards.append(episode_reward)

    # Calculate summary statistics
    avg_reward = np.mean(total_rewards)
    avg_final_value = np.mean(portfolio_values)
    success_rate = np.sum(np.array(total_rewards) > 0) / len(total_rewards) * 100

    if verbose:
        print(f"\nEvaluation Results (over {episodes} episodes):")
        print(f"Average Total Reward: {avg_reward:.4f}")
        print(f"Average Final Portfolio Value: ${avg_final_value:.2f}")
        print(f"Success Rate (positive return): {success_rate:.2f}%")

    return total_rewards, portfolio_values


def backtest_portfolio(model, data_path, max_stocks=100, lookback=30, initial_cash=10000, years=2, verbose=True):
    """Run a comprehensive backtest of the portfolio strategy with option to silence output"""
    if verbose:
        print(f"Running {years}-year backtest simulation...")

    # Initialize the environment with the backtest data
    backtest_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=years
    )

    backtest_env = PortfolioTradingEnv(backtest_iter, initial_cash=initial_cash)

    # Run backtest
    obs, _ = backtest_env.reset()
    done = False

    portfolio_values = [initial_cash]
    returns = []
    cash_allocations = []
    transaction_costs = []
    holdings_diversification = []
    daily_turnover = []

    step = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        # Track portfolio composition
        cash_allocations.append(action[0])

        # Number of stocks with allocation > 1%
        significant_holdings = sum(1 for a in action[1:] if a > 0.01)
        holdings_diversification.append(significant_holdings)

        # Execute step
        obs, reward, done, _, info = backtest_env.step(action)

        # Track metrics
        portfolio_values.append(info["portfolio_value"])
        if step > 0:
            returns.append((portfolio_values[-1] / portfolio_values[-2]) - 1)
        transaction_costs.append(info["transaction_costs"])

        # Calculate turnover (sum of absolute changes in allocation)
        if step == 0:
            daily_turnover.append(0)
        else:
            turnover = info["transaction_costs"] / (portfolio_values[-2] * backtest_env.transaction_cost)
            daily_turnover.append(turnover)

        step += 1

        # Print progress
        if verbose and step % 50 == 0:
            print(f"Backtest step {step}, Portfolio value: ${portfolio_values[-1]:.2f}")

    # Calculate performance metrics
    metrics = calculate_portfolio_metrics(portfolio_values)

    backtest_results = {
        "portfolio_values": portfolio_values,
        "returns": returns,
        "cash_allocations": cash_allocations,
        "transaction_costs": transaction_costs,
        "holdings_diversification": holdings_diversification,
        "daily_turnover": daily_turnover,
        "metrics": metrics,
        "steps": step,
        "final_value": portfolio_values[-1]
    }

    return backtest_results


# Example usage:
if __name__ == "__main__":
    # Set up parameters for tuning
    DATA_PATH ="/content/drive/MyDrive/historical_data.csv"
    TUNING_DIR = "/content/tuning_models"

    # First round of hyperparameter tuning with reduced parameter space
    tuning_results = hyperparameter_tuning(
        DATA_PATH,
        base_model_path=TUNING_DIR,
        max_stocks_options=[50, 100],
        lookback_options=[15, 30],
        learning_rate_options=[1e-4, 3e-4],
        gamma_options=[0.99],
        buffer_size_options=[100000],
        action_noise_options=["ou"],
        noise_std_options=[0.1, 0.2],
        timesteps=100000,  # Reduced for initial tuning
        eval_episodes=3
    )

    # Get the path to the tuning results
    results_files = [f for f in os.listdir(os.path.join(TUNING_DIR, "results")) if f.startswith("tuning_results")]
    latest_results = sorted(results_files)[-1]
    results_path = os.path.join(TUNING_DIR, "results", latest_results)

    # Fine-tune the best model by Sharpe ratio
    fine_tuned_results = fine_tune_best_model(
        DATA_PATH,
        results_path,
        best_model_type="best_by_sharpe",
        timesteps=500000,  # Full training for the best model
        eval_episodes=10
    )

Tuning with 16 parameter combinations


Hyperparameter tuning:   0%|          | 0/16 [00:00<?, ?it/s]


Training model 1/16: ddpg_model_1_ms50_lb15_lr0.0001_g0.99_ns0.1
Loading data from /content/drive/MyDrive/historical_data.csv...
Adding technical indicators...
Technical indicators added successfully
Scaling features...
Preparing training data...


Output()

/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

...

Step 1000: Reward = 0.008224, Portfolio Value: 192977.47, Transaction Costs: 45.23

Step 1100: Reward = -0.006171, Portfolio Value: 252461.14, Transaction Costs: 56.50

Step 1200: Reward = 0.028384, Portfolio Value: 377273.91, Transaction Costs: 456.82

Step 1300: Reward = -0.001066, Portfolio Value: 397328.07, Transaction Costs: 84.65

Step 1400: Reward = -0.004063, Portfolio Value: 767130.89, Transaction Costs: 60.85